In [12]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras import models
from tensorflow.keras.models import Sequential
import pickle
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report,confusion_matrix
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from tensorflow.keras.layers import TextVectorization,Embedding,LSTM,Dense,Dropout,Bidirectional
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.optimizers import Adam
import re

In [13]:
data=pd.read_csv('/content/drive/MyDrive/finalLargeDs.csv')

In [3]:
def clean_text(text):
    # Convert to string
    text = str(text)
    # Remove URLs, mentions, special characters (keep letters and spaces)
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'[^a-zA-Z\s\u0600-\u06FF]', '', text)  # Keep English/Arabic letters
    text = text.lower()
    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()
    return text

In [ ]:
data['Text'] = data['clean_comment'].apply(clean_text)

In [4]:
model = models.load_model('/content/drive/MyDrive/BEsafe_models/BesageLargedataModel.h5')

In [5]:
with open('/content/drive/MyDrive/BesafeLArgedataTokenize.pkl', 'rb') as f:
    tokenizer = pickle.load(f)

In [6]:
def predict_threat(text, model, tokenizer, threshold=0.5):
    cleaned = clean_text(text)
    seq = tokenizer.texts_to_sequences([cleaned])
    vec =pad_sequences(seq,maxlen=100,padding='post')
    prob = model.predict(vec)[0][0]
    label = int(prob > threshold)
    return label, prob

In [8]:
while True:
    user_input = input("Enter a sentence (or 'exit'): ")

    if user_input.lower() == "exit":
        break

    result = predict_threat(user_input,model,tokenizer)
    label, prob = result
    label = "Threat" if label == 1 else "Non-Threat"
    print(f"Text: {user_input}\nPreediction: {label} (probability: {prob:.4f})")

Enter a sentence (or 'exit'): Help me I am being chased by a stranger
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 67ms/step
Text: Help me I am being chased by a stranger
Preediction: Threat (probability: 0.9136)
Enter a sentence (or 'exit'): She said she was chased by a bear and was caught by it
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
Text: She said she was chased by a bear and was caught by it
Preediction: Non-Threat (probability: 0.0006)
Enter a sentence (or 'exit'): he took the wrong turn when he crashed into the wall
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step
Text: he took the wrong turn when he crashed into the wall
Preediction: Non-Threat (probability: 0.0049)
Enter a sentence (or 'exit'): Someone should help me please there is a man with a gun outside my house
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 63ms/step
Text: Someone should help me please there is a man with a gun outside my house
Preediction: Threat (probability: 0.5962)
Enter a sentence (or 'exit'): Someone should help me please there is a man with a gun 

In [ ]:
# Assuming your test data is in a DataFrame
# Example: df_test with a column 'Text'
# df_test = pd.read_csv("your_test_file.csv")

def batch_predict(df, model, tokenizer, text_column="Text", output_file="predictions.csv"):
    """
    Predicts threat vs non-threat for all rows in df and saves results to a CSV.

    Parameters:
    - df: pandas DataFrame with text data
    - model: trained Keras model
    - tokenizer: fitted tokenizer
    - text_column: name of column containing the text
    - output_file: path to save predictions CSV
    """
    # Tokenize and pad sequences
    sequences = tokenizer.texts_to_sequences(df[text_column])
    # If you have used pad_sequences with maxlens
    padded = pad_sequences(sequences, maxlen=model.input_shape[1], padding='post', truncating='post')

    # Make predictions
    probs = model.predict(padded, verbose=1)

    # For binary classification: threshold at 0.5
    labels = (probs > 0.5).astype(int).flatten()

    # Map to readable text
    df["Prediction"] = ["Threat" if l == 1 else "Non-Threat" for l in labels]
    df["Probability"] = probs.flatten()

    # Save to CSV
    df.to_csv(output_file, index=False)
    print(f"Predictions saved to {output_file}")
    return df

In [15]:
data

,Text,Label
0,b' conflict of interest note by your user name...,0
1,b'update actually i changed this to something ...,0
2,b'mrca article sniperz thanks for your comment...,0
3,b'arguing that bart and caltrain should get mo...,0
4,b' blocked hi i blocked you for hours for bein...,0
...,...,...
125099,Let\'s have some fun my way.,1
125100,Heartbroken.,0
125101,Sit down or I\'ll throw you off the bus.,1
125102,Leave me alone.,0


In [ ]:
pd.read_csv()